# W2D1 — Understand the Data — Lab

**Week 2 · Day 1 · Data Engineering** · Lab

A shop's order system has been exporting to CSV for eighteen months, and nobody has checked the
export. Today you open it for the first time. You will not clean it and you will not model it —
you will find out **what is in it and what is wrong with it**, because every decision you make for
the rest of the week depends on what you learn in the next 115 minutes. The question behind the
whole week is whether an order will be **returned**, and by the end of today you will know how
often that happens and roughly what the data will let you say about it. You leave with
`data_quality_notes.md`, the list of problems D3 has to fix.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٢ اليوم ١ — فهم البيانات

**الأسبوع ٢ · اليوم ١ · هندسة البيانات** · معمل عملي

يُصدِّر نظام طلبات أحد المتاجر ملفات CSV منذ ثمانية عشر شهرًا، ولم يفحص أحد هذا التصدير. واليوم
تفتحه لأول مرة. لن تنظّف البيانات ولن تبني نموذجًا — بل ستكتشف **ما الذي فيها وما الخطأ فيها**،
لأن كل قرار تتّخذه في بقية الأسبوع يعتمد على ما تتعلّمه في المئة وخمس عشرة دقيقة القادمة. والسؤال
الذي يقف خلف الأسبوع كله هو: هل سيُرجَع الطلب (`returned`)؟ وبنهاية اليوم ستعرف كم مرة يحدث ذلك،
وما الذي تسمح لك البيانات بقوله عنه تقريبًا. وستخرج بملف `data_quality_notes.md`، وهو قائمة
المشكلات التي على معمل اليوم الثالث إصلاحها.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Describe a dataset's shape, columns and dtypes, and say which columns are numeric and which are categorical.
- Measure missing values, duplicate rows and inconsistent category spellings, and say how much of each there is.
- Recognise that one text column holds several date formats, and explain why that breaks naive parsing.
- Find values that are impossible rather than merely unusual, and argue why they are impossible.
- State the class balance of the `returned` target and explain why a 23% positive rate matters.
- Turn a question about the data into a small `groupby` investigation and interpret the answer.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- وصف شكل مجموعة البيانات وأعمدتها وأنواعها، وتحديد الأعمدة الرقمية والأعمدة الفئوية.
- قياس القيم المفقودة والصفوف المكرّرة وتضارب تهجئة الفئات، وتحديد مقدار كل منها.
- إدراك أن عمودًا نصيًّا واحدًا يحمل عدة صيغ للتاريخ، وشرح سبب إفساد ذلك للتحليل الساذج.
- إيجاد القيم المستحيلة لا الغريبة فحسب، وتبرير استحالتها.
- ذكر توازن الفئات في هدف `returned` وشرح أهمية نسبة إيجابية تبلغ ٢٣٪.
- تحويل سؤال عن البيانات إلى تحقيق صغير باستخدام `groupby` وتفسير نتيجته.

</div>

## About the data

**Dataset:** `messy_sales` — built for this course · CC0 · 5,000 rows

One row is one sales order: when it was placed and shipped, which city and channel it came through,
the customer's loyalty tier, how many units at what price, and what the shop eventually recognised
as revenue. The column the week is about is **`returned`** — 1 if the customer sent the order back,
0 if they kept it. Predicting returns is worth doing because a return costs the shop the shipping
both ways plus the handling, so knowing which orders are at risk changes how you pack, price and
promise them.

**Watch out:** the whole file is the gotcha. It was built broken on purpose — missing values in
three different patterns, four date formats in one column, duplicated rows, fourteen spellings of
five cities, and **two columns that leak a target**. You are not expected to find all of that
today. You are expected to find *some* of it, and to write down what you find.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `messy_sales` — أُعدّت لهذه الدورة · رخصة CC0 · ٥٠٠٠ صف

الصف الواحد هو طلب بيع واحد: متى قُدّم ومتى شُحن، ومن أي مدينة وقناة جاء، ومستوى ولاء العميل، وكم
وحدة بأي سعر، وما الإيراد الذي اعترف به المتجر في النهاية. والعمود الذي يدور حوله الأسبوع هو
**`returned`** — ويساوي ١ إذا أعاد العميل الطلب و٠ إذا احتفظ به. والتنبّؤ بالإرجاع يستحق العناء
لأن الإرجاع يكلّف المتجر الشحن في الاتجاهين إضافة إلى المناولة، فمعرفة الطلبات المعرّضة للخطر
تغيّر طريقة التغليف والتسعير والوعود.

**انتبه:** الملف كله مشكلة. صُمّم معطوبًا عن قصد — قيم مفقودة بثلاثة أنماط مختلفة، وأربع صيغ
للتاريخ في عمود واحد، وصفوف مكرّرة، وأربع عشرة تهجئة لخمس مدن، و**عمودان يُسرّبان الهدف**. ولا
يُتوقّع منك أن تجد كل ذلك اليوم، بل أن تجد **بعضه** وأن تدوّن ما وجدت.

</div>

## Setup

Run the cell below first. It installs anything missing, fixes the random seed, and finds the
dataset — whether you are on your own machine or on Google Colab.

<div dir="rtl" align="right">

## الإعداد

شغّل الخلية التالية أولًا. تُثبّت ما ينقص، وتُثبّت البذرة العشوائية، وتجد ملف البيانات — سواء كنت
على جهازك أو على Google Colab.

</div>

In [1]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("pandas")                         # this lab is pandas only — no modelling today
seed_everything(42)                      # course-wide seed

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

DATA = get_dataset("messy_sales")
print(describe_dataset("messy_sales"))
print("\n", versions(), "| device:", device())

⚠️  Checksum mismatch for messy_sales.csv
  expected: fc71cf3e578ac8a93c823491b70487f461857eefd5ec50348f487677aedd23fb
  actual:   a75cb5f8a8ef10446e07b33eec2df00b2afb12f2642483d57b842b85628d09ae
The file may be corrupt or the registry may be out of date.
📁 messy_sales: using cached file at C:\Users\pkupr\AIEP_Olo_student\shared\data_cache\messy_sales.csv
messy_sales  (5000 rows, 0.41 MB)
  Source:  Built for this course
  Licence: CC0 (classroom fixture, generated)
  Target:  returned

  EN: Deliberately broken data: missing values in three different patterns, dates in four formats, 120 duplicated rows, fourteen spellings of five cities, and two columns that leak a target. Built so every cleaning technique in W2D3 has something real to fix, and so one dataset carries the whole Week 2 story from data quality through EDA to imbalanced classification. `returned` is 23% positive and genuinely hard — honest features reach AUC ~0.66, while the leaked column reaches ~0.999. Generated by tool

## Section 1 — Warm-up  (≈25 min)

Guided. The code below works. Run it, read it, then change the one parameter each task points at
and observe what moves. Nothing here is a puzzle — the point is to get the file open and to
remember the pandas verbs you will need all week.

<div dir="rtl" align="right">

## القسم الأول — التهيئة (نحو ٢٥ دقيقة)

قسم موجَّه. الشيفرة أدناه تعمل. شغّلها واقرأها، ثم غيّر المعامل الذي تشير إليه كل مهمة ولاحظ ما
يتغيّر. لا يوجد لغز هنا — الهدف هو فتح الملف واستعادة أفعال pandas التي ستحتاجها طوال الأسبوع.

</div>

In [2]:
# Working code. Nothing to fill in here.
# Note we read with pandas' defaults — no parse_dates, no dtype hints. Seeing what
# pandas guesses on its own is the first piece of evidence about the file.
raw = pd.read_csv(DATA)

print(f"rows: {len(raw):,}   columns: {raw.shape[1]}")
raw.head()

rows: 5,000   columns: 13


,order_id,order_date,ship_date,city,channel,customer_tier,quantity,unit_price,discount_pct,commission_paid,refund_amount,revenue,returned
0,102389,2025-06-12,2025-06-21,DAMMAM,online,silver,32,53.17,0.244,38.32,0.00,1277.35,0
1,102810,15 Sep 2024,2024-09-18,Riyadh,online,bronze,26,112.91,0.198,72.12,1177.01,2403.99,1
2,101306,2025/04/24,2025-05-08,TABUK,online,bronze,27,33.67,0.192,21.52,0.00,717.36,0
3,104508,2024-04-15,2024-04-16,riyadh,store,gold,9,14.78,0.293,2.83,0.00,94.25,0
4,102377,18 Feb 2025,2025-03-02,Abha,online,silver,13,16.49,0.061,5.98,0.00,199.28,0


### Task 1.1 — Change what you look at

`head()` shows the first five rows, which is the least representative sample in the file — early
rows are often the cleanest. Change `n` below, and swap `.head` for `.sample`. Keep `random_state`
fixed so your neighbour sees the same rows you do.

**Look for:** a column whose values do not all look like they were written by the same system.

<div dir="rtl" align="right">

### المهمة ١٫١ — غيّر ما تنظر إليه

تُظهر `head()` أول خمسة صفوف، وهي أقل عيّنة تمثيلًا في الملف — فالصفوف الأولى غالبًا أنظفها. غيّر
قيمة `n` أدناه، واستبدل `.head` بـ `.sample`، مع تثبيت `random_state` كي يرى زميلك الصفوف نفسها.

**ابحث عن:** عمود لا تبدو قيمه كلها مكتوبة بالنظام نفسه.

</div>

In [4]:
n = 10

raw.sample(n, random_state=0)

,order_id,order_date,ship_date,city,channel,customer_tier,quantity,unit_price,discount_pct,commission_paid,refund_amount,revenue,returned
398,103985,2024-01-20,2024-02-02,Jeddah,store,bronze,2,60.95,0.204,2.86,0.00,95.44,0
3833,100269,2024-06-03,2024-06-06,Riyadh,phone,bronze,7,71.15,NaN,14.90,0.00,496.77,0
4836,100522,2024/12/07,2024-12-09,Jeddah,store,bronze,34,25.27,0.218,19.97,0.00,665.78,0
4572,102768,06/18/2024,2024-06-28,Dammam,online,bronze,30,23.67,0.076,19.64,174.98,654.67,1
636,100031,02/01/2025,2025-02-12,Dammam,online,bronze,36,97.88,0.079,100.00,1712.95,3333.36,1
2545,102517,2025/01/18,2025-01-20,Riyadh,phone,silver,24,45.46,NaN,21.84,0.00,727.93,0
1161,102744,2024/09/21,2024-10-04,Dammam,phone,silver,10,13.53,NaN,2.87,0.00,95.64,0
2230,100306,04/07/2024,2024-04-12,Jeddah,phone,bronze,37,15.64,NaN,10.62,0.00,354.08,0
148,103523,06/08/2025,2025-06-22,Riyadh,phone,gold,9,61.46,NaN,9.99,0.00,332.86,0
2530,103790,2024/06/24,2024-07-02,DAMMAM,online,silver,36,47.07,0.097,47.71,0.00,1590.43,0


### Task 1.2 — Ask what pandas decided

`dtypes` is pandas' guess, not the truth. A column of dates that pandas calls `object` is a column
of *text* that happens to contain dates — and text does not sort, subtract or compare like a date.

**Look for:** which columns came back as `object`, and ask yourself for each one whether that is
what you would have chosen.

<div dir="rtl" align="right">

### المهمة ١٫٢ — اسأل عمّا قرّره pandas

`dtypes` هو تخمين pandas لا الحقيقة. فعمود تواريخ يسمّيه pandas `object` هو عمود **نص** يصادف أنه
يحتوي تواريخ — والنص لا يُرتَّب ولا يُطرح ولا يُقارن مثل التاريخ.

**ابحث عن:** الأعمدة التي عادت بنوع `object`، واسأل نفسك عن كل منها: هل هذا ما كنت ستختاره؟

</div>

In [5]:
# Working code. Read the two lists it prints.
numeric_cols = raw.select_dtypes(include="number").columns.tolist()
object_cols = raw.select_dtypes(include="object").columns.tolist()

print("pandas thinks these are numeric: ", numeric_cols)
print("pandas thinks these are text:    ", object_cols)
print()
raw.dtypes.to_frame("dtype")

pandas thinks these are numeric:  ['order_id', 'quantity', 'unit_price', 'discount_pct', 'commission_paid', 'refund_amount', 'revenue', 'returned']
pandas thinks these are text:     ['order_date', 'ship_date', 'city', 'channel', 'customer_tier']



C:\Users\pkupr\AppData\Local\Temp\ipykernel_9024\948717021.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = raw.select_dtypes(include="object").columns.tolist()


,dtype
order_id,int64
order_date,str
ship_date,str
city,str
channel,str
customer_tier,str
quantity,int64
unit_price,float64
discount_pct,float64
commission_paid,float64


## Section 2 — Core  (≈60 min)

This is the lab. Each task has a goal; you write the code. You are building evidence, not fixing
anything — resist the urge to clean as you go. D3 is the cleaning lab, and it will go faster if
today's notes are precise.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي (نحو ٦٠ دقيقة)

هذا هو صلب المعمل. لكل مهمة هدف، وأنت من يكتب الشيفرة. أنت تجمع أدلّة ولا تصلح شيئًا — قاوم
الرغبة في التنظيف أثناء العمل. فمعمل اليوم الثالث هو معمل التنظيف، وسيكون أسرع إذا كانت ملاحظات
اليوم دقيقة.

</div>

### Task 2.1 — Measure what is missing

A count of nulls is not useful on its own; a *rate* is. Produce one table with, for each column,
the number of missing values and the share of rows they represent, sorted worst first, showing
only the columns that actually have some.

**Then answer, in the markdown cell after:** one column is far worse than the others. Is that
column missing because nobody filled it in, or is something about *which* rows go missing?

<div dir="rtl" align="right">

### المهمة ٢٫١ — قِس ما هو مفقود

عدد القيم الفارغة وحده غير مفيد، بل **النسبة** هي المفيدة. أنتج جدولًا واحدًا يبيّن لكل عمود عدد
القيم المفقودة وحصتها من الصفوف، مرتّبًا من الأسوأ، ومقتصرًا على الأعمدة التي فيها نقص فعلًا.

**ثم أجب في خلية Markdown التالية:** هناك عمود أسوأ من البقية بكثير. هل هو مفقود لأن أحدًا لم
يملأه، أم أن هناك شيئًا مشتركًا بين **الصفوف** التي تغيب فيها القيمة؟

</div>

In [6]:
# ────────────────────────────────────────────────────────────────────
# 1) Ask the frame which cells are null, then add them up per column.
# 2) A rate is that count divided by the number of rows — put both in one table.
# 3) Sort so the worst column is at the top, and drop the columns with nothing missing.
# Search: "pandas count missing values per column percentage"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html
#
# ١) اسأل الإطار عن الخلايا الفارغة، ثم اجمعها لكل عمود.
# ٢) النسبة هي ذلك العدد مقسومًا على عدد الصفوف — ضع الاثنين في جدول واحد.
# ٣) رتّب بحيث يكون أسوأ عمود في الأعلى، واحذف الأعمدة التي لا نقص فيها.
# ابحث عن: "pandas count missing values per column percentage"
# ────────────────────────────────────────────────────────────────────

missing = pd.DataFrame({
    "count": raw.isna().sum(),
    "rate": raw.isna().mean() * 100
})
missing = (
    missing[missing["count"] > 0]
    .sort_values("rate", ascending=False)
)
print(missing.to_string(formatters={"rate": "{:.1f}%".format}))


              count  rate
discount_pct   1473 29.5%


In [7]:
# ────────────────────────────────────────────────────────────────────
# 1) You suspect the missingness is not random. Test it: group the rows by a
#    categorical column and ask what share of each group is missing.
# 2) Try `channel` first. One group should stand out completely.
# Search: "pandas groupby mean of isna"
#
# ١) تشكّ في أن النقص ليس عشوائيًّا. اختبر ذلك: جمّع الصفوف حسب عمود فئوي
#    واسأل عن نسبة النقص في كل مجموعة.
# ٢) ابدأ بعمود `channel`. ستبرز إحدى المجموعات تمامًا.
# ابحث عن: "pandas groupby mean of isna"
# ────────────────────────────────────────────────────────────────────

by_channel = (
    raw["discount_pct"]
    .isna()
    .groupby(raw["channel"])
    .mean()
    .sort_values(ascending=False)
)
print("discount_pct missing rate by channel:")
print(by_channel.to_string(float_format="{:.1%}".format))

discount_pct missing rate by channel:
channel
phone    100.0%
store     17.7%
online    17.5%


**Write your answer here.** Is `discount_pct` missing at random? What did the breakdown by
`channel` tell you, and what would go wrong if you filled every gap with the column's average?

*(Replace this sentence with two or three of your own.)*

<div dir="rtl" align="right">

**اكتب إجابتك هنا.** هل `discount_pct` مفقود عشوائيًّا؟ ماذا أخبرك التوزيع حسب `channel`، وما الذي
سيحدث لو ملأت كل فجوة بمتوسّط العمود؟

*(استبدل هذه الجملة بجملتين أو ثلاث من عندك.)*

</div>

<div dir="rtl" align="right">

### إجابتي
* إن القيم المفقودة في `discount_pct` ماهي عشوائية لان كل الطلبات الي بphone مفقود منها الخصم 
* ولو حطينا مكانها المتوسط  بيصير قيم خصم وهمية لطلبات الهاتف ويبعد العلاقة بين فقدان البيانات وقناة البيع

</div>

### Task 2.2 — Count the duplicates, then ask whether they matter

Find how many rows are exact duplicates of another row. Then — and this is the part people skip —
check whether removing them would move the data. Compare the mean of `revenue` and the rate of
`returned` before and after dropping them.

A duplicate set that changes the target rate is a different problem from one that does not.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — عُدّ المكرّرات ثم اسأل هل تهمّ

جِد كم صفًّا يكرّر صفًّا آخر تمامًا. ثم — وهذا ما يتخطّاه الناس — تحقّق ممّا إذا كان حذفها سيحرّك
البيانات. قارن متوسّط `revenue` ونسبة `returned` قبل الحذف وبعده.

فمجموعة مكرّرات تغيّر نسبة الهدف مشكلة مختلفة عن مجموعة لا تغيّرها.

</div>

In [8]:
# ────────────────────────────────────────────────────────────────────
# 1) The frame can tell you which rows duplicate an earlier one. Count them.
# 2) Build the de-duplicated frame too, but do NOT overwrite `raw` — today is
#    about evidence, and D3 needs the original to clean.
# 3) Compare row count, mean revenue and mean returned across the two.
# Search: "pandas duplicated drop_duplicates"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html
#
# ١) يستطيع الإطار إخبارك بالصفوف التي تكرّر صفًّا سابقًا. عُدّها.
# ٢) ابنِ الإطار منزوع التكرار أيضًا، لكن لا تستبدل `raw` — فاليوم للأدلّة،
#    ومعمل اليوم الثالث يحتاج الأصل لتنظيفه.
# ٣) قارن عدد الصفوف ومتوسّط revenue ومتوسّط returned بين الاثنين.
# ابحث عن: "pandas duplicated drop_duplicates"
# ────────────────────────────────────────────────────────────────────

n_duplicates = int(raw.duplicated().sum())
deduplicated = raw.drop_duplicates()
comparison = pd.DataFrame(
    {
        "rows": [len(raw), len(deduplicated)],
        "mean_revenue": [
            raw["revenue"].mean(),
            deduplicated["revenue"].mean()
        ],
        "return_rate": [
            raw["returned"].mean(),
            deduplicated["returned"].mean()
        ]
    },
    index=["before", "after"]
)
print(f"{n_duplicates} exact duplicate rows\n")
print(comparison.to_string(float_format="{:.4f}".format))
print(f"{n_duplicates} exact duplicate rows\n")
print(comparison.to_string(float_format="{:.4f}".format))

120 exact duplicate rows

        rows  mean_revenue  return_rate
before  5000      823.4627       0.2308
after   4880      823.0218       0.2299
120 exact duplicate rows

        rows  mean_revenue  return_rate
before  5000      823.4627       0.2308
after   4880      823.0218       0.2299


### Task 2.3 — Find the inconsistent categories

Three columns are categorical: `city`, `channel` and `customer_tier`. For each, list the distinct
values and how often each occurs.

Two of the three are clean. One is not — and the damage is invisible until you look at the exact
strings. Count how many distinct values `city` has, and how many actual cities you think that is.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — جِد الفئات غير المتّسقة

ثلاثة أعمدة فئوية: `city` و`channel` و`customer_tier`. اعرض لكل منها القيم المميّزة وتكرار كل قيمة.

اثنان من الثلاثة نظيفان، وواحد ليس كذلك — والضرر غير مرئي حتى تنظر إلى النصوص بدقّة. عُدّ القيم
المميّزة في `city`، وكم مدينة فعلية تظنّها.

</div>

In [9]:
# ────────────────────────────────────────────────────────────────────
# 1) Loop over the three categorical columns and print the value counts of each.
# 2) Wrap each value in quotes when you print it — the problem is whitespace and
#    capitalisation, and neither is visible without a delimiter.
# 3) For `city`, also print how many distinct values there are.
# Search: "pandas value_counts"
#
# ١) مُرّ على الأعمدة الفئوية الثلاثة واطبع تكرار القيم لكل منها.
# ٢) ضع كل قيمة بين علامتَي اقتباس عند الطباعة — فالمشكلة مسافات وحالة أحرف،
#    ولا يظهر أي منهما دون فاصل.
# ٣) اطبع أيضًا عدد القيم المميّزة في `city`.
# ابحث عن: "pandas value_counts"
# ────────────────────────────────────────────────────────────────────

CATEGORICAL = ["channel", "customer_tier", "city"]
for column in CATEGORICAL:
    print(f"\n{column}:")
    
    counts = raw[column].value_counts(dropna=False)
    for value, count in counts.items():
        print(f'"{value}": {count}')

print(f"\nNumber of distinct city values: {raw['city'].nunique()}")


channel:
"online": 2729
"store": 1551
"phone": 720

customer_tier:
"bronze": 2449
"silver": 1785
"gold": 766

city:
"Riyadh": 1214
"Jeddah": 920
"Dammam": 610
"Tabuk": 419
"Abha": 366
"riyadh": 269
"Riyadh ": 264
" Jeddah": 192
"JEDDAH": 188
"abha": 141
"DAMMAM": 133
"dammam ": 124
"tabuk": 82
"TABUK ": 78

Number of distinct city values: 14


### Task 2.4 — Prove the date column is not one format

`order_date` is text. Your job is to show it holds **more than one** written format, and to say how
many of each.

Do not parse it yet. Classify the raw strings by their *shape* — a regular expression per format
you spot — and count how many rows match each. Print any row that matches none; that is how you
find the format you missed.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — أثبت أن عمود التاريخ ليس بصيغة واحدة

`order_date` نص. ومهمتك أن تُظهر أنه يحمل **أكثر من صيغة** مكتوبة، وأن تذكر عدد كل صيغة.

لا تحلّله إلى تاريخ بعد. صنّف النصوص الخام حسب **شكلها** — تعبير نمطي لكل صيغة تلاحظها — وعُدّ
الصفوف المطابقة لكل منها. واطبع أي صف لا يطابق أي صيغة؛ هكذا تجد الصيغة التي فاتتك.

</div>

In [11]:
# ────────────────────────────────────────────────────────────────────
# 1) Look at a sample of the raw strings first — 15 is enough to see the variety.
# 2) Write one regular expression per shape you see. Think in terms of "four digits,
#    dash, two digits" rather than in terms of day and month.
# 3) Count the rows matching each pattern, then count the rows matching none.
# Search: "pandas Series str.match regex"
# https://pandas.pydata.org/docs/reference/api/pandas.Series.str.match.html
#
# ١) انظر أولًا إلى عيّنة من النصوص الخام — خمسة عشر تكفي لرؤية التنوّع.
# ٢) اكتب تعبيرًا نمطيًّا لكل شكل تراه. فكّر بمنطق «أربعة أرقام ثم شرطة ثم رقمان»
#    لا بمنطق اليوم والشهر.
# ٣) عُدّ الصفوف المطابقة لكل نمط، ثم عُدّ الصفوف غير المطابقة لأي نمط.
# ابحث عن: "pandas Series str.match regex"
# ────────────────────────────────────────────────────────────────────

print(raw["order_date"].sample(15, random_state=0).tolist())

patterns = {
    "YYYY-MM-DD": r"^\d{4}-\d{2}-\d{2}$",
    "YYYY/MM/DD": r"^\d{4}/\d{2}/\d{2}$",
    "MM/DD/YYYY": r"^\d{2}/\d{2}/\d{4}$",
    "D Mon YYYY": r"^\d{1,2} [A-Za-z]{3} \d{4}$"
}

matches = {
    name: raw["order_date"].str.match(pattern, na=False)
    for name, pattern in patterns.items()
}

shape_counts = {
    name: int(mask.sum())
    for name, mask in matches.items()
}

matched_any = pd.Series(False, index=raw.index)
for mask in matches.values():
    matched_any = matched_any | mask

for name, count in shape_counts.items():
    print(f"    {name:12s} {count:5d} rows")

print(f"\n    matching none: {int((~matched_any).sum())}")

['2024-01-20', '2024-06-03', '2024/12/07', '06/18/2024', '02/01/2025', '2025/01/18', '2024/09/21', '04/07/2024', '06/08/2025', '2024/06/24', '03/21/2025', '03/26/2025', '01/09/2025', '1 Feb 2025', '23 May 2025']
    YYYY-MM-DD    1317 rows
    YYYY/MM/DD    1204 rows
    MM/DD/YYYY    1225 rows
    D Mon YYYY    1254 rows

    matching none: 0


### Task 2.5 — Separate the impossible from the merely unusual

Run `describe()` on the numeric columns and read it properly: for each column ask what the minimum
and the maximum would mean in the real shop.

Then find the rows that are **impossible** rather than surprising. A large order is surprising. An
order that shipped before it was placed is impossible, and no amount of domain knowledge makes it
fine. Count how many rows break at least one rule you can justify.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — افصل المستحيل عن الغريب فحسب

شغّل `describe()` على الأعمدة الرقمية واقرأها كما ينبغي: اسأل عن كل عمود ماذا تعني قيمته الدنيا
والعليا في المتجر الحقيقي.

ثم جِد الصفوف **المستحيلة** لا المفاجئة. فالطلب الكبير مفاجئ، أما الطلب الذي شُحن قبل أن يُقدَّم
فمستحيل، ولا تجعله أي معرفة بالمجال مقبولًا. عُدّ الصفوف التي تخرق قاعدة واحدة على الأقل تستطيع
تبريرها.

</div>

In [12]:
# ────────────────────────────────────────────────────────────────────
# 1) Describe the numeric columns and read the min and max of each.
# 2) To compare the two dates you must parse them. `format="mixed"` handles a
#    column holding several formats — this is a look, not the real cleaning.
# 3) Write one boolean expression per rule you can defend, then count the rows
#    that break any of them.
# Search: "pandas to_datetime format mixed"
#
# ١) صِف الأعمدة الرقمية واقرأ القيمة الدنيا والعليا لكل منها.
# ٢) لمقارنة التاريخين عليك تحليلهما. ويتعامل `format="mixed"` مع عمود يحمل عدة
#    صيغ — وهذه نظرة لا تنظيف حقيقي.
# ٣) اكتب تعبيرًا منطقيًّا لكل قاعدة تستطيع الدفاع عنها، ثم عُدّ الصفوف
#    التي تخرق أيًّا منها.
# ابحث عن: "pandas to_datetime format mixed"
# ────────────────────────────────────────────────────────────────────

print(raw.describe().to_string(float_format="{:.2f}".format))
print()

parsed_order = pd.to_datetime(
    raw["order_date"],
    format="mixed",
    errors="coerce"
)

parsed_ship = pd.to_datetime(
    raw["ship_date"],
    format="mixed",
    errors="coerce"
)

rule_breaks = {
    "shipped before ordered": parsed_ship < parsed_order,
    "non-positive quantity": raw["quantity"] <= 0,
    "non-positive unit price": raw["unit_price"] <= 0,
    "future order date": parsed_order > pd.Timestamp.today().normalize()
}

impossible_any = pd.DataFrame(rule_breaks).any(axis=1)
n_impossible = int(impossible_any.sum())

for name, mask in rule_breaks.items():
    print(f"    {name:26s} {int(mask.sum()):5d} rows")

print(f"\nRows breaking at least one rule: {n_impossible}")

       order_id  quantity  unit_price  discount_pct  commission_paid  refund_amount  revenue  returned
count   5000.00   5000.00     5000.00       3527.00          5000.00        5000.00  5000.00   5000.00
mean  102445.32     20.86       47.71          0.15            24.70         115.77   823.46      0.23
std     1408.71     11.65       37.46          0.08            26.10         358.73   870.04      0.42
min   100001.00      1.00        3.09          0.00             0.14           0.00     4.50      0.00
25%   101229.75     11.00       23.57          0.09             7.73           0.00   257.61      0.00
50%   102450.50     21.00       37.59          0.14            16.91           0.00   563.69      0.00
75%   103662.25     31.00       59.11          0.20            32.34           0.00  1077.94      0.00
max   104880.00     40.00      375.87          0.45           360.03        4821.61 12000.89      1.00

    shipped before ordered         0 rows
    non-positive quantity     

### Task 2.6 — Meet the target

Now look at `returned`, the column the rest of the week predicts. Report how many orders were
returned, how many were kept, and the **positive rate** — the share of orders that came back.

Then compute the score a model would get by always predicting "kept". Write that number down. It
is the number every model you build this week has to beat, and in D5 you will find out that
beating it is not the same as being useful.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — تعرّف على الهدف

انظر الآن إلى `returned`، العمود الذي تتنبّأ به بقية الأسبوع. اذكر كم طلبًا أُرجع، وكم طلبًا احتُفظ
به، و**النسبة الإيجابية** — أي حصة الطلبات التي عادت.

ثم احسب الدقّة التي سيحقّقها نموذج يتنبّأ دائمًا بـ«لم يُرجَع». دوّن هذا الرقم. فهو الرقم الذي على
كل نموذج تبنيه هذا الأسبوع أن يتجاوزه، وستكتشف في اليوم الخامس أن تجاوزه ليس مرادفًا للفائدة.

</div>

In [13]:
# ────────────────────────────────────────────────────────────────────
# 1) Count the two classes, and get the share of each.
# 2) The positive rate is the mean of a 0/1 column — that is why the encoding is useful.
# 3) A model that always predicts the majority class is right exactly as often as that
#    class occurs. Compute that share.
# Search: "pandas value_counts normalize"
#
# ١) عُدّ الفئتين واحسب حصة كل منهما.
# ٢) النسبة الإيجابية هي متوسّط عمود من صفر وواحد — ولهذا فائدة هذا الترميز.
# ٣) النموذج الذي يتنبّأ دائمًا بالفئة الأكبر يصيب بقدر تكرار تلك الفئة
#    تمامًا. احسب حصتها.
# ابحث عن: "pandas value_counts normalize"
# ────────────────────────────────────────────────────────────────────

class_counts = raw["returned"].value_counts().sort_index()
class_shares = raw["returned"].value_counts(normalize=True).sort_index()
return_rate = raw["returned"].mean()
majority_accuracy = class_shares.max()
print(f"kept     (0): {class_counts[0]:,} ({class_shares[0]:.1%})")
print(f"returned (1): {class_counts[1]:,} ({class_shares[1]:.1%})")
print(f"\npositive rate: {return_rate:.1%}")
print(f"always predicting 'kept' would be {majority_accuracy:.1%} accurate")

kept     (0): 3,846 (76.9%)
returned (1): 1,154 (23.1%)

positive rate: 23.1%
always predicting 'kept' would be 76.9% accurate


### Task 2.7 — Your own investigation

You have a target and a set of columns. Form **one** hypothesis about what makes a return more
likely, and test it with a `groupby`.

Pick something you can defend from how shops actually work — for example, that orders which took
longer to ship come back more often, or that a particular channel is worse than the others. State
the hypothesis in the markdown cell, test it in the code cell, and then say whether the data
supported it.

You are not proving causation here and you are not building a model. You are practising the loop:
*question → measurement → interpretation.*

<div dir="rtl" align="right">

### المهمة ٢٫٧ — تحقيقك الخاص

لديك هدف ومجموعة أعمدة. كوّن **فرضية واحدة** عمّا يجعل الإرجاع أرجح، واختبرها بـ`groupby`.

اختر شيئًا تستطيع الدفاع عنه من واقع عمل المتاجر — مثلًا أن الطلبات التي طال شحنها تعود أكثر، أو
أن قناة بعينها أسوأ من غيرها. اذكر الفرضية في خلية Markdown، واختبرها في خلية الشيفرة، ثم قل هل
أيّدتها البيانات.

أنت لا تثبت سببية هنا ولا تبني نموذجًا، بل تتدرّب على الحلقة: **سؤال ← قياس ← تفسير.**

</div>

**My hypothesis:** *(write it here before you run anything — one sentence, and say which way you
expect the effect to go.)*

<div dir="rtl" align="right">

**فرضيتي:** *(اكتبها هنا قبل تشغيل أي شيء — جملة واحدة، مع ذكر الاتجاه الذي تتوقّعه للأثر.)*

</div>

<div dir="rtl" align="right">


### فرضيتي:

 أتوقع  يكون الإرجاع في الطلبات من الإنترنت أعلى من بقية قنوات البيع لأن العميل ما يقدر يشوف المنتج  قبل يشتري

</div>

In [14]:
# ────────────────────────────────────────────────────────────────────
# 1) Turn your hypothesis into a column you can group by. If it is about shipping
#    speed, you need the number of days between the two dates first.
# 2) A continuous column has to be bucketed before it can be grouped — cut it into
#    a handful of bins.
# 3) Group by that column and take the mean of `returned`. That mean IS the return
#    rate of the group, which is why the 0/1 encoding pays off.
# 4) Print the group sizes next to the rates — a rate over 12 rows is not evidence.
# Search: "pandas groupby agg mean count"
#
# ١) حوّل فرضيتك إلى عمود تستطيع التجميع حسبه. فإن كانت عن سرعة الشحن فأنت تحتاج
#    عدد الأيام بين التاريخين أولًا.
# ٢) العمود المتّصل يجب تقسيمه إلى فئات قبل التجميع — قسّمه إلى
#    بضع سلال.
# ٣) جمّع حسب ذلك العمود وخذ متوسّط `returned`. فهذا المتوسّط **هو** نسبة الإرجاع
#    في المجموعة، وهنا تظهر فائدة ترميز صفر وواحد.
# ٤) اطبع أحجام المجموعات بجوار النسب — فنسبة محسوبة على اثني عشر صفًّا ليست دليلًا.
# ابحث عن: "pandas groupby agg mean count"
# ────────────────────────────────────────────────────────────────────

hypothesis_result = (
    raw.groupby("channel")
    .agg(
        orders=("returned", "size"),
        return_rate=("returned", "mean")
    )
    .sort_values("return_rate", ascending=False)
)

print(hypothesis_result.to_string(
    formatters={"return_rate": "{:.1%}".format}
))

         orders return_rate
channel                    
online     2729       28.3%
phone       720       23.1%
store      1551       13.9%


**What I found:** *(one or two sentences. Did the data support your hypothesis? If the effect is
there, is it big enough to be worth anything? If the group sizes are uneven, say so.)*

<div dir="rtl" align="right">

**ما وجدته:** *(جملة أو جملتان. هل أيّدت البيانات فرضيتك؟ وإن وُجد الأثر، فهل هو كبير بما يكفي
ليستحق شيئًا؟ وإن كانت أحجام المجموعات متفاوتة فاذكر ذلك.)*

</div>


<div dir="rtl" align="right">


### ما وجدته:

 البيانات تتفق مع فرضيتي لان نسبة الإرجاع في الطلبات عبر الإنترنت 28.3% مقارنةً بـ23.1% للهاتف و13.9% للمتجر الفرق بين الإنترنت والمتجر كبير ورغم تفاوت المجموعات كل مجموعة تحتوي عدد من الطلبات للمقارنة 
</div>

## Section 3 — Stretch  (≈30 min)

Open-ended, and lower expectation of completeness.

Two of the columns in this file could not have been known at the moment the order was placed. Find
them, and for each one write a sentence explaining *when* its value becomes available.

You are not being asked to prove they are dangerous — that is D3's job, and it has a name for what
they are. You are being asked to practise the question that finds them: **"at the instant I would
need to make this prediction, would I actually have this number?"**

**Link to your capstone:** the first thing to do with your own dataset is exactly this. Before you
model anything, list every column and write down when its value becomes known. Columns that arrive
after the thing you are predicting are the single most common reason a capstone model scores
beautifully and fails in the demo.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي (نحو ٣٠ دقيقة)

قسم مفتوح، ولا يُتوقّع إكماله بالكامل.

هناك عمودان في هذا الملف لم يكن من الممكن معرفتهما لحظة تقديم الطلب. جِدهما، واكتب لكل منهما جملة
تشرح **متى** تصبح قيمته متاحة.

لا يُطلب منك إثبات خطورتهما — فتلك مهمة اليوم الثالث، وله اسم لما هما عليه. بل يُطلب منك التدرّب
على السؤال الذي يجدهما: **«في اللحظة التي أحتاج فيها إلى هذا التنبّؤ، هل سيكون هذا الرقم بحوزتي
فعلًا؟»**

**الصلة بمشروعك:** أول ما تفعله بمجموعة بياناتك هو هذا بالضبط. قبل أن تبني أي نموذج، اسرد كل عمود
ودوّن متى تصبح قيمته معروفة. فالأعمدة التي تصل بعد الشيء الذي تتنبّأ به هي السبب الأول لأن يحقّق
مشروع التخرّج نتيجة رائعة ثم يفشل في العرض.

</div>

In [15]:
# ────────────────────────────────────────────────────────────────────
# There is no single right answer to write in code here. Look at the column list,
# pick the ones that describe something that happens AFTER an order is placed, and
# check your intuition by seeing how they behave for kept versus returned orders.
#
# لا توجد إجابة واحدة صحيحة تُكتب شيفرةً هنا. انظر إلى قائمة الأعمدة، واختر تلك التي
# تصف شيئًا يحدث **بعد** تقديم الطلب، وتحقّق من حدسك برؤية سلوكها في الطلبات
# المحتفظ بها مقابل المُرجعة.
# ────────────────────────────────────────────────────────────────────

print("Columns:")
print(raw.columns.tolist())

suspicious_comparison = (
    raw.groupby("returned")[["refund_amount", "revenue"]]
    .agg(["mean", "median", "min", "max"])
)

print("\nSuspicious columns by returned class:")
print(suspicious_comparison.to_string(float_format="{:.2f}".format))

Columns:
['order_id', 'order_date', 'ship_date', 'city', 'channel', 'customer_tier', 'quantity', 'unit_price', 'discount_pct', 'commission_paid', 'refund_amount', 'revenue', 'returned']

Suspicious columns by returned class:
         refund_amount                     revenue                     
                  mean median  min     max    mean median  min      max
returned                                                               
0                 0.02   0.00 0.00   54.91  772.00 527.09 4.50 12000.89
1               501.54 286.61 0.00 4821.61  994.98 732.27 8.01  8446.68


**The columns I would not have had, and when they arrive:**

1. *(column — when its value becomes known)*
2. *(column — when its value becomes known)*

<div dir="rtl" align="right">

**الأعمدة التي لم تكن ستتوفّر لديّ، ومتى تصل:**

١. *(العمود — متى تصبح قيمته معروفة)*
٢. *(العمود — متى تصبح قيمته معروفة)*

</div>





<div dir="rtl" align="right">


### الأعمدة التي لم تكن ستتوفّر لديّ، ومتى تصل:

1. `refund_amount` — تصبح قيمته معروفة بعد اذا رجع العميل الطلب تتم معالجة مبلغ الاسترداد.
2. `revenue` — تصبح قيمته النهائية معروفة بعد اكتمال الطلب والإرجاع وتسجيل الإيراد الفعلي.

</div>

## Save your artefact

`data_quality_notes.md` is the handover to D3. It records what you measured today, with the actual
numbers, so tomorrow's cleaning is driven by evidence instead of memory. D2 reads it too, as the
list of things worth plotting.

<div dir="rtl" align="right">

## احفظ مخرجاتك

ملف `data_quality_notes.md` هو التسليم إلى اليوم الثالث. يسجّل ما قِسته اليوم بالأرقام الفعلية،
فيقود التنظيف غدًا بالأدلّة لا بالذاكرة. ويقرؤه اليوم الثاني أيضًا بوصفه قائمة بما يستحق الرسم.

</div>

In [17]:
date_formats_text = "\n".join(
    f"- {name}: {count:,} rows"
    for name, count in shape_counts.items()
)

channel_missing_text = "\n".join(
    f"- {channel}: {rate:.1%}"
    for channel, rate in by_channel.items()
)

report_text = f"""# Data Quality Notes

## Dataset
- Rows: {len(raw):,}
- Columns: {raw.shape[1]}

## Missing values
- `discount_pct`: {int(missing.loc["discount_pct", "count"]):,} missing values
- Missing rate: {missing.loc["discount_pct", "rate"]:.1f}%

Missing rate by channel:
{channel_missing_text}

The missingness is not random because all phone orders are missing `discount_pct`.

## Duplicate rows
- Exact duplicate rows: {n_duplicates:,}
- Rows before removal: {len(raw):,}
- Rows after removal: {len(deduplicated):,}
- Return rate before: {raw["returned"].mean():.2%}
- Return rate after: {deduplicated["returned"].mean():.2%}

## Inconsistent categories
- `city` contains {raw["city"].nunique()} written values representing 5 actual cities.
- The differences include capitalisation and extra whitespace.

## Date formats
`order_date` contains four written formats:

{date_formats_text}

- Rows matching no known format: {int((~matched_any).sum())}

## Impossible values
- Rows breaking at least one tested rule: {n_impossible}
- Tested rules: shipping before ordering, non-positive quantity, non-positive price, and future order dates.

## Target
- Kept orders: {class_counts[0]:,} ({class_shares[0]:.1%})
- Returned orders: {class_counts[1]:,} ({class_shares[1]:.1%})
- Positive return rate: {return_rate:.1%}
- Majority baseline accuracy: {majority_accuracy:.1%}

## Investigation
Online orders had the highest return rate at {hypothesis_result.loc["online", "return_rate"]:.1%},
compared with {hypothesis_result.loc["phone", "return_rate"]:.1%} for phone orders and
{hypothesis_result.loc["store", "return_rate"]:.1%} for store orders.

## Columns unavailable at order time
- `refund_amount`: known after a return is processed.
- `revenue`: final value known after the order and any return are processed.
"""

out = ARTEFACT_DIR / "data_quality_notes.md"
out.write_text(report_text, encoding="utf-8")

print(f"Saved {out}\n")
print(out.read_text(encoding="utf-8"))

Saved c:\Users\pkupr\AIEP_Olo_student\week_2_data_engineering\labs_v2\D1_understand_the_data\artefacts\data_quality_notes.md

# Data Quality Notes

## Dataset
- Rows: 5,000
- Columns: 13

## Missing values
- `discount_pct`: 1,473 missing values
- Missing rate: 29.5%

Missing rate by channel:
- phone: 100.0%
- store: 17.7%
- online: 17.5%

The missingness is not random because all phone orders are missing `discount_pct`.

## Duplicate rows
- Exact duplicate rows: 120
- Rows before removal: 5,000
- Rows after removal: 4,880
- Return rate before: 23.08%
- Return rate after: 22.99%

## Inconsistent categories
- `city` contains 14 written values representing 5 actual cities.
- The differences include capitalisation and extra whitespace.

## Date formats
`order_date` contains four written formats:

- YYYY-MM-DD: 1,317 rows
- YYYY/MM/DD: 1,204 rows
- MM/DD/YYYY: 1,225 rows
- D Mon YYYY: 1,254 rows

- Rows matching no known format: 0

## Impossible values
- Rows breaking at least one tested ru

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخلية أخيرًا. كل فحص يفشل يخبرك بما يجب إصلاحه ولماذا.

</div>

In [18]:
# --- Sanity checks ----------------------------------------------------------------

check(len(raw) == 5000 and raw.shape[1] == 13,
      f"messy_sales should load as 5000 x 13, got {raw.shape[0]} x {raw.shape[1]}",
      f"يجب أن تُحمّل messy_sales بحجم ٥٠٠٠ × ١٣، والقيمة الحالية {raw.shape[0]} × {raw.shape[1]}")

check(0.20 < return_rate < 0.26,
      f"the returned rate should be about 23%, you measured {return_rate:.1%}",
      f"يجب أن تكون نسبة الإرجاع نحو ٢٣٪، وقد قِست {return_rate:.1%}")

check(n_duplicates > 0,
      f"there are duplicate rows in this file — you counted {n_duplicates}",
      f"يوجد صفوف مكرّرة في هذا الملف — وقد عددت {n_duplicates}")

check(raw["city"].nunique() > 5,
      f"city should have more spellings than real cities; you found {raw['city'].nunique()}",
      f"يجب أن تزيد تهجئات city عن عدد المدن الحقيقية، وقد وجدت {raw['city'].nunique()}")

check(sum(1 for c in shape_counts.values() if c > 0) >= 3,
      f"order_date should hold at least 3 written formats, you classified "
      f"{sum(1 for c in shape_counts.values() if c > 0)}",
      f"يجب أن يحمل order_date ثلاث صيغ مكتوبة على الأقل، وقد صنّفت "
      f"{sum(1 for c in shape_counts.values() if c > 0)}")

check(missing.loc["discount_pct", "rate"] > 10,
      f"discount_pct should be more than 10% missing, you measured "
      f"{missing.loc['discount_pct', 'rate']:.1f}%",
      f"يجب أن يتجاوز النقص في discount_pct ١٠٪، وقد قِست "
      f"{missing.loc['discount_pct', 'rate']:.1f}%")

check((ARTEFACT_DIR / "data_quality_notes.md").exists(),
      "data_quality_notes.md should exist in artefacts/ — D3 reads it tomorrow",
      "يجب أن يوجد data_quality_notes.md في artefacts/ — يقرؤه اليوم الثالث غدًا")

report()

──────────────────────────────────────────────────────────────────
  ✓  messy_sales should load as 5000 x 13, got 5000 x 13
  ✓  the returned rate should be about 23%, you measured 23.1%
  ✓  there are duplicate rows in this file — you counted 120
  ✓  city should have more spellings than real cities; you found 14
  ✓  order_date should hold at least 3 written formats, you classified 4
  ✓  discount_pct should be more than 10% missing, you measured 29.5%
  ✓  data_quality_notes.md should exist in artefacts/ — D3 reads it tomorrow
──────────────────────────────────────────────────────────────────
  ✅ All 7 checks passed. / اجتزت جميع الفحوصات (7).
──────────────────────────────────────────────────────────────────


## What's next

Tomorrow (D2) you take the same file and the same target and **look** at it: the class balance as a
chart, the return rate broken down by every categorical column you catalogued today, and the
relationships between the numeric ones. Today you found out what is wrong with the data; tomorrow
you find out what it has to say.

<div dir="rtl" align="right">

## ماذا بعد

غدًا (اليوم الثاني) تأخذ الملف نفسه والهدف نفسه و**تنظر** إليه: توازن الفئات كرسم بياني، ونسبة
الإرجاع موزّعة حسب كل عمود فئوي فهرسته اليوم، والعلاقات بين الأعمدة الرقمية. اليوم عرفت ما الخطأ
في البيانات، وغدًا تعرف ما الذي تقوله.

</div>